# Asistente de Histología (ETL & Chat)
Este notebook te permite inicializar el entorno, clonar tu proyecto estructural de Github, instalar dependencias de sistema y de Python, ingestar los PDFs y correr tu CLI interactivo de inferencia RAG.
Asegúrate de configurar tu sesión de Colab con un acelerador de hardware **GPU (ej. T4)**.

In [1]:
import os

# 1. Instalar pre-requisitos de sistema operativo (vitales para OCR e Imágenes) y UV para rapidez
!sudo apt-get update -qq && sudo apt-get install -y poppler-utils tesseract-ocr -qq
!pip install uv -q

# 2. Clonar el repositorio usando el token de Github en Secretos de Colab
from google.colab import userdata
github_token = userdata.get('GITHUB_TOKEN')
github_user = "tompivel"   # <-- Reemplaza por tu usuario/repo si hace falta
repo_name = "histo-test"
repo_url = f"https://{github_token}@github.com/{github_user}/{repo_name}.git"

if not os.path.exists(repo_name):
    !git clone -b refactoring --single-branch {repo_url} -q

%cd {repo_name}

# 3. Instalar dependencias puras mediante UV basándose en el pyproject.toml
!uv pip install --system -r pyproject.toml

import nest_asyncio
nest_asyncio.apply()

# 4. RESURRECT 'imp' TO FIX AUTORELOAD IN PYTHON 3.12
!uv pip install --system zombie-imp

# 5. This tells Colab to watch your .py files for changes and
# automatically reload them before executing any new code.
%load_ext autoreload
%autoreload 2

print("✅ Entorno preparado exitosamente.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package poppler-utils.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━

### Paso 1: Ejecutar Ingesta (Data ETL)
Analiza PDFs, aplica OCR, calcula vectores Texto, UNI y PLIP y rellena Neo4J. *(Solo correr cuando debas actualizar el Graph Database)*.

In [3]:
from core.ingestion import IngestionPipeline
from utils.config import userdata
import os

async def run_etl():
    print("Levantando entorno de Data Pipeline...")
    pipeline = IngestionPipeline(
        neo4j_uri=userdata.get('NEO4J_URI'),
        neo4j_user=userdata.get('NEO4J_USERNAME'),
        neo4j_pass=userdata.get('NEO4J_PASSWORD'),
        device='cuda'
    )
    await pipeline.initialize()

    # Generamos la carpeta de escucha si no existe:
    directorio_pdfs = os.path.join(os.getcwd(), 'pdf')
    os.makedirs(directorio_pdfs, exist_ok=True)
    print(f"Coloca tus manuales PDF en: {directorio_pdfs} antes de continuar.\n")

    # force=True si quieres sobrescribir datos
    await pipeline.execute(pdfs_dir=directorio_pdfs, force=False)
    await pipeline.close()

# -----------------------------------------------------
#  Si deseas poblar Neo4j, descomenta la siguiente línea:
# -----------------------------------------------------
await run_etl()

Levantando entorno de Data Pipeline...
🚀 Pipeline ETL Initialized on Backend: CUDA


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Connecting to Neo4j DB and loading local models...
✅ Neo4j Connected: neo4j+s://30af270f.databases.neo4j.io
🏗️ Creating Neo4j Schema (v4.4 UNI + PLIP)...
✅ Neo4j Schema Ready (3 indexes)
🔄 Loading UNI (MahmoodLab)...
✅ UNI Loaded
🔄 Loading PLIP (vinid/plip)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: vinid/plip
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ PLIP Loaded
Coloca tus manuales PDF en: /content/histo-test/pdf antes de continuar.

📋 Extracting and inferring the general syllabus...
📋 Extracting syllabus...
✅ Syllabus 34 topics extracted
📄 Indexing text chunks in Neo4j...
  arch2.pdf: 49 chunks
📸 Extracting and Indexing PDF images (Vision Models)...
📂 Extracting from 1 PDFs...
  📸 35 images processed from arch2.pdf
✅ Total images extracted: 35
🔗 Building K-NN visual similarity relations...
🔗 Creating :SIMILAR_A relations (UNI method, threshold=0.85)...
✅ 59 :SIMILAR_A relations linked
✅ ETL indexing completed successfully.


### Paso 2: Chat Interactivo (RAG)
Instancia la red nodal en LangGraph. Podrás escribir textos directo desde el Colab, o enviarle path a imágenes subidas al entorno `/content/`.

In [4]:
from core.cli import interactive_mode

# Inicia el bucle eterno de consulta conversacional
await interactive_mode()


🔬 ASISTENTE DE HISTOLOGÍA - RAG MULTIMODAL GROQ (v4.4) 🔬
Usando Llama-4-17B, LangGraph, UNI + PLIP, Neo4j, Qdrant
------------------------------------------------------------
Comandos:
  'salir'    -> Terminar la sesión
  'imagen <ruta>' -> Subir una imagen junto con tu próxima pregunta (Ej: imagen /tmp/foto.jpg)
    
🖥️ Initializing agent on: CUDA
✅ Neo4j Connected: neo4j+s://30af270f.databases.neo4j.io
🧠 Initializing Central LLM (Groq: Llama-4-17B)...
🧠 Initializing Local Text Embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Loading UNI (MahmoodLab)...
✅ UNI Loaded
🔄 Loading PLIP (vinid/plip)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: vinid/plip
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ PLIP Loaded
✅ Base models ready.
🏗️ Creating Neo4j Schema (v4.4 UNI + PLIP)...
✅ Neo4j Schema Ready (3 indexes)
✅ LangGraph compiled (v4.4)

👤 Tu consulta: imagen imagenes_chat/unrelated.jpg
✅ Imagen 'unrelated.jpg' cargada en memoria temporal.

[📷 Imagen cargada y lista para la siguiente pregunta]

👤 Tu consulta: ¿podrías explicarme las características de la imagen?
⏳ Analizando y procesando (LangGraph en curso)...
   🔍 [Node] Verifying domain...
   🔄 Pre-calculating Semantic Anchor embeddings...
   📐 Semantic Similarity to domain: 0.8102
   👁️ [Node] Processing user image...
   🧠 [Node] Retrieving memory context...
   🔎 [Node] Executing hybrid multispace search...
   📊 Hybrid Search: Txt=8 | UNI=8 | PLIP=8 | Ent=0 | Vec=14 -> 35
   ✍️ [Node] Generating final response (Role: Sceptical Judge)...
   🧠 Interaction saved to semantic memory (Session: cli_loca)

🧠 ASISTENTE:
Lo siento, pero la imagen subida por el usuario no corresponde a una imagen histológica. Por lo tanto, no puedo pro